In [99]:
import pandas as pd
import numpy as np
import re


from unicodedata import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering, DBSCAN


# Clusterización de versiones

Ejemplo sólo para la marca especificada

In [152]:
path = "data/cleaned.csv"
df = pd.read_csv(path)

brand = "Volkswagen"
model = "Taos"

version_df = df[(df["Marca"] == brand) & (df["Modelo"] == model)].reset_index(drop=True)

### Limpieza y preprocesamiento de versiones

Este código se encarga de limpiar y normalizar los nombres de las versiones de vehículos para facilitar su posterior análisis y agrupación:

**Función** `clean_version()`
- **Normalización de caracteres**: Convierte texto a minúsculas y elimina acentos/caracteres especiales usando Unicode NFKD
- **Filtrado de caracteres**: Solo mantiene letras, números y espacios, eliminando símbolos y puntuación
- **Limpieza de espacios**: Normaliza espacios múltiples a espacios únicos y elimina espacios al inicio/final
- **Manejo de valores nulos**: Convierte valores `NaN` a strings vacíos


In [153]:
def clean_version(s: str) -> str:
    s = "" if pd.isna(s) else s
    s = normalize("NFKD", s.lower()).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

version_df["version_clean"] = version_df["Versión"].apply(clean_version)
version_df = version_df[version_df["version_clean"] != ""].reset_index(drop=True)

print("Número de versiones únicas:", version_df["version_clean"].nunique())

print("\nPrimeras 5 versiones únicas:")
print(version_df["version_clean"].value_counts()[:5])

Número de versiones únicas: 24

Primeras 5 versiones únicas:
version_clean
250 comfortline              261
250 highline                 255
250 highline bi tono          58
250 highline tiptronic        36
250 comfortline tiptronic     29
Name: count, dtype: int64


## HAC y TF-IDF

In [154]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"])

# Filter out zero vectors before clustering
non_zero_mask = np.array(X.sum(axis=1)).flatten() > 0
X_filtered = X[non_zero_mask].toarray()
taos_df_filtered = version_df[non_zero_mask].copy()

model = AgglomerativeClustering(
    metric="cosine",
    linkage="average",
    distance_threshold=0.35,
    n_clusters=None,
)

# Fit on filtered data
clusters_filtered = model.fit_predict(X_filtered)

# Assign clusters back to original dataframe
version_df["cluster"] = -1  # Default for zero vectors
version_df.loc[non_zero_mask, "cluster"] = clusters_filtered

In [155]:
canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary

,cluster,version_canon,count
5,5,250 Comfortline,263
0,0,250 Highline,258
2,2,250 Highline Tiptronic,66
8,8,250 Highline Bi Tono,60
14,14,250 Hero,15
13,13,250 Highline Bi Tono Tiptronic,7
4,4,Highline 250 bi Tono,4
1,1,250 COMFORTLINE TIPTRONIC L23,2
3,3,Highline 250,2
6,6,Highline 250Tsi,2


## DBSCAN

- Leer `CountVectorizer` en [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) (podemos jugar con los argumentos)

In [156]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"]).toarray()

print(X.shape)

(686, 222)


In [158]:
db = DBSCAN(eps=0.3, min_samples=10, metric="cosine")
version_df["cluster"] = db.fit_predict(X)

canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
summary


,cluster,version_canon,count
2,1,250 Highline,373
1,0,250 Comfortline,297
3,2,250 Hero,15
0,-1,250 HERO TIPTRONIC L21,1
